# Tag 07 — Anfänger
## Your First Decision Tree — Heart Disease

تمرین‌های این نوت‌بوک:
1. آموزش `DecisionTreeClassifier(max_depth=3)` و چاپ Accuracy برای Train/Test
2. رسم درخت و پیدا کردن Root Feature
3. مقایسه معیارهای `gini` و `entropy`

In [ ]:
import sys
!{sys.executable} -m pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay

# Find project root automatically
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name not in ["Tag_07_Decision_Trees_Project", ""] and not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "01_anfaenger"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Output dir:", OUTPUT_DIR)


## 1) Load data

In [ ]:

heart_path = DATA_DIR / "heart.csv"
if not heart_path.exists():
    raise FileNotFoundError(f"Could not find {heart_path}. Put heart.csv into data/raw/.")

df = pd.read_csv(heart_path)
print(df.shape)
df.head()


In [ ]:

print(df.info())
print("\nMissing values:")
print(df.isna().sum())
print("\nTarget distribution:")
print(df["target"].value_counts(normalize=True).round(3))


## 2) Split X/y and Train/Test

In [ ]:

X = df.drop(columns=["target"])
y = df["target"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_tr.shape)
print("X_test :", X_te.shape)
print("Train target ratio:\n", y_tr.value_counts(normalize=True).round(3))
print("Test target ratio:\n", y_te.value_counts(normalize=True).round(3))


## 3) Task 1 — Train Decision Tree max_depth=3

In [ ]:

dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tr, y_tr)

train_acc = dt.score(X_tr, y_tr)
test_acc = dt.score(X_te, y_te)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy : {test_acc:.4f}")

pd.DataFrame({"metric": ["train_accuracy", "test_accuracy"], "value": [train_acc, test_acc]}).to_csv(
    OUTPUT_DIR / "beginner_accuracy.csv", index=False
)


In [ ]:

y_pred = dt.predict(X_te)
print(classification_report(y_te, y_pred, target_names=["No disease", "Disease"]))

cm = confusion_matrix(y_te, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["No disease", "Disease"])
disp.plot()
plt.title("Confusion Matrix — Beginner Decision Tree")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "beginner_confusion_matrix.png", dpi=150)
plt.show()


## 4) Task 2 — Visualise the Tree and Root Feature

In [ ]:

fig, ax = plt.subplots(figsize=(18, 8))
plot_tree(
    dt,
    feature_names=X.columns.tolist(),
    class_names=["No", "Yes"],
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax
)
plt.title("Decision Tree — max_depth=3")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "beginner_decision_tree.png", dpi=150)
plt.show()

root_feature_index = dt.tree_.feature[0]
root_feature = X.columns[root_feature_index]
print("Root feature:", root_feature)

print("\nText version of the tree:\n")
print(export_text(dt, feature_names=list(X.columns)))


## 5) Task 3 — Compare Gini vs Entropy

In [ ]:

results = []
models = {}
for criterion in ["gini", "entropy"]:
    model = DecisionTreeClassifier(max_depth=3, criterion=criterion, random_state=42)
    model.fit(X_tr, y_tr)
    train_acc = model.score(X_tr, y_tr)
    test_acc = model.score(X_te, y_te)
    root = X.columns[model.tree_.feature[0]]
    results.append({
        "criterion": criterion,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "root_feature": root,
        "node_count": model.tree_.node_count,
        "max_depth": model.tree_.max_depth
    })
    models[criterion] = model

results_df = pd.DataFrame(results)
results_df.to_csv(OUTPUT_DIR / "beginner_gini_vs_entropy.csv", index=False)
results_df


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(22, 7))
for ax, criterion in zip(axes, ["gini", "entropy"]):
    plot_tree(
        models[criterion],
        feature_names=X.columns.tolist(),
        class_names=["No", "Yes"],
        filled=True,
        rounded=True,
        fontsize=7,
        ax=ax
    )
    ax.set_title(f"criterion={criterion}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "beginner_gini_vs_entropy_trees.png", dpi=150)
plt.show()


## 6) What to send back to ChatGPT

بعد از اجرا، این فایل‌ها/اسکرین‌شات‌ها را بده تا تحلیل کنم:

- `outputs/01_anfaenger/beginner_accuracy.csv`
- `outputs/01_anfaenger/beginner_gini_vs_entropy.csv`
- عکس درخت یا خروجی `Root feature`
- خروجی `classification_report`